# Day 12, Step 4 — Context Offloading (agent)

**Problem:** Step 3 compresses context by dropping whole sections. But some information is too important to drop and too large to send in full — a long log, a long document.

**Context offloading** solves this differently: write the large result somewhere outside the model's context (here, a plain dict standing in for a file, vector store, or database), keep only a short reference and summary in the context that's actually sent, and read the full text back only if the request specifically needs it. In this version, the **model** — not a keyword check — decides whether the full text is needed, from a fixed allowlist (`FULL` or `SUMMARY`), with `SUMMARY` as the safe default: when uncertain, don't over-share.

**Graph:** `START -> offload_raw_result -> build_context -> decide_detail_level -> (agent decision) -> [retrieve_full_text ->] ask_model -> END`

## Setup

In [13]:
from typing import TypedDict

import ollama
from langgraph.graph import END, START, StateGraph

MODEL = "gpt-oss:120b-cloud"
OFFLOAD_KEY = "log-4471"

# Stand-in for external storage (a file, vector store, or database in a real system).
OFFLOAD_STORE = {}


## State

`detail_decision` holds the model's own choice (`FULL` or `SUMMARY`) — the field the routing function reads.

In [14]:
class State(TypedDict):
    request: str
    raw_result: str
    summary: str
    detail_decision: str
    context: str
    answer: str


## `offload_raw_result` — write the large result out

Stores the full `raw_result` in `OFFLOAD_STORE` under a fixed key and keeps only a short summary in state — the full text itself is never put into `context`.

In [15]:
def offload_raw_result(state: State):
    OFFLOAD_STORE[OFFLOAD_KEY] = state["raw_result"]
    summary = state["raw_result"][:100] + "..."
    return {"summary": summary}


## `build_context` — assemble the small version

Builds `context` from the task and the summary only, plus a note showing how large the full text is and where it's stored — not the full text itself.

In [16]:
def build_context(state: State):
    lines = [
        f"TASK: {state['request']}",
        f"SUMMARY: {state['summary']}",
        f"NOTE: full text stored at '{OFFLOAD_KEY}' "
        f"({len(state['raw_result'])} chars, not sent unless needed).",
    ]
    return {"context": "\n".join(lines)}


## `decide_detail_level` — the agent decision

Asks the model whether the request needs the exact/full/verbatim text or whether the summary is enough. The allowed answers are a fixed pair, `FULL` or `SUMMARY` — anything else is corrected to `SUMMARY`, the safe default that avoids sending more than necessary.

In [17]:
def decide_detail_level(state: State):
    prompt = (
        "Does answering this request need the exact, full, verbatim text, or is a short "
        "summary enough?\n"
        "Return only FULL or SUMMARY.\n\n"
        f"Request: {state['request']}\n"
        f"Summary available: {state['summary']}"
    )
    response = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    decision = response["message"]["content"].strip().upper()
    if decision not in ("FULL", "SUMMARY"):
        decision = "SUMMARY"
    return {"detail_decision": decision}


## `route_after_decision` — where the agent boundary lives

Reads `detail_decision` and sends execution to `retrieve_full_text` or straight to `ask_model` — the model's own output is what LangGraph routes on.

In [18]:
def route_after_decision(state: State):
    if state["detail_decision"] == "FULL":
        return "retrieve_full_text"
    return "ask_model"


## `retrieve_full_text` — read the offloaded text back

Only reached when the model decided `FULL`. Reads the full text back out of `OFFLOAD_STORE` and appends it to the context — the one time the full text actually gets sent.

In [19]:
def retrieve_full_text(state: State):
    full_text = OFFLOAD_STORE[OFFLOAD_KEY]
    context = state["context"] + f"\nFULL TEXT: {full_text}"
    return {"context": context}


## `ask_model`

Sends whatever `context` ended up being built — summary-only, or summary plus full text.

In [20]:
def ask_model(state: State):
    response = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": state['context']}],
    )
    return {"answer": response["message"]["content"]}


## Build the graph

`decide_detail_level` routes conditionally via `add_conditional_edges` to either `retrieve_full_text` (then `ask_model`) or straight to `ask_model`.

In [21]:
graph_builder = StateGraph(State)
graph_builder.add_node("offload_raw_result", offload_raw_result)
graph_builder.add_node("build_context", build_context)
graph_builder.add_node("decide_detail_level", decide_detail_level)
graph_builder.add_node("retrieve_full_text", retrieve_full_text)
graph_builder.add_node("ask_model", ask_model)

graph_builder.add_edge(START, "offload_raw_result")
graph_builder.add_edge("offload_raw_result", "build_context")
graph_builder.add_edge("build_context", "decide_detail_level")
graph_builder.add_conditional_edges("decide_detail_level", route_after_decision)
graph_builder.add_edge("retrieve_full_text", "ask_model")
graph_builder.add_edge("ask_model", END)
graph = graph_builder.compile()


## Run it

The sample request asks for the "exact wording" of the last error — watch `detail_decision` in the printed context: a model reading the request carefully should decide `FULL`, and the context should include the `FULL TEXT` line. Try a vaguer request (like "what happened in the log?") and re-run to see the `SUMMARY` path instead.

In [22]:
result = graph.invoke(
    {
        "request": "Give me the exact wording of the first error in the log",
        "raw_result": (
            "2026-09-24 09:14:02 INFO connection established\n"
            "2026-09-24 09:14:05 WARN retry attempt 1\n"
            "2026-09-24 09:14:09 ERROR connection reset by peer, session terminated"
        ),
    }
)
print("Context sent to the model:\n", result["context"])
print("\nAnswer:\n", result["answer"])


Context sent to the model:
 TASK: Give me the exact wording of the first error in the log
SUMMARY: 2026-09-24 09:14:02 INFO connection established
2026-09-24 09:14:05 WARN retry attempt 1
2026-09-24 ...
NOTE: full text stored at 'log-4471' (159 chars, not sent unless needed).
FULL TEXT: 2026-09-24 09:14:02 INFO connection established
2026-09-24 09:14:05 WARN retry attempt 1
2026-09-24 09:14:09 ERROR connection reset by peer, session terminated

Answer:
 The first error in the log is:

**2026-09-24 09:14:09 ERROR connection reset by peer, session terminated**
